# Технологии ИИ · Занятие 3 · Практика и домашнее задание

**Фамилия, имя:** _(Недиев Турпал)_ · **Группа:** ДИБ 401 · **Срок:** до следующей пары

Файл `auth_log.csv` должен лежать в той же папке, что и этот блокнот. Ячейки
запускаются по порядку: курсор в ячейку → **Shift + Enter**. Под каждой ячейкой
есть вопрос, ответ пишите словами в ячейку «Ответ» (два щелчка по ней —
редактирование, Shift + Enter — сохранить).

Три части:

| Часть | Что | Откуда |
|---|---|---|
| **А** | Журнал входов как таблица: запустить, прочитать, ответить | практика занятий 2–3 |
| **Б** | Пять задач ИБ → тип задачи машинного обучения | теория занятия 3 |
| **В** | Домашняя практика: изменить код под новый вопрос | новое |

Как сдавать — в самом конце блокнота.

## Если библиотек ещё нет

1. VS Code → Select Kernel (справа вверху) → Python Environments → ваш Python 3.11+.
   Предложит установить `ipykernel` — соглашайтесь.
2. Выполните ячейку ниже. Если интернета нет, а рядом с блокнотом лежит папка
   `wheels`, замените команду на `%pip install --no-index --find-links=wheels pandas scikit-learn`.
3. Подробная инструкция с картинками ошибок — `ИНСТРУКЦИЯ-настройка.pdf` в папке.

In [31]:
%pip install pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\nikitos\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


---
# Часть А. Журнал входов как таблица

## А1. Среда

**Ячейка А1.** Версия Python: нужна 3.11 или новее.

In [32]:
import sys
print(sys.version)

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


**Ячейка А2.** Библиотеки на месте: две версии, без ошибок.

In [33]:
import pandas as pd
import sklearn
print(pd.__version__, sklearn.__version__)

3.0.5 1.9.1


## А2. Открываем журнал

**Ячейка А3.** Загрузить файл и показать первые 5 строк.

In [34]:
df = pd.read_csv("auth_log.csv")
df.head()

,timestamp,user,ip,result
0,2025-03-10 08:55:00,ivanov,10.0.1.11,failure
1,2025-03-10 09:00:00,ivanov,10.0.1.11,success
2,2025-03-10 09:11:00,ivanov,10.0.1.11,success
3,2025-03-10 09:22:00,ivanov,10.0.1.11,success
4,2025-03-10 09:33:00,ivanov,10.0.1.11,success


**Вопрос А3.** Что записано в одной строке таблицы? Какие колонки могут стать признаками? Есть ли в файле метка «атака / норма»?

**Ответ:** _(время, имя, айпи адрес и результат входа.Да, признаком может стать ip и время (ночью или днем). Нету.)_

**Ячейка А4.** Размер таблицы: (строк, колонок).

In [35]:
df.shape

(69, 4)

**Вопрос А4.** Сколько примеров в выборке? Много это или мало для настоящего журнала организации?

**Ответ:** _(69 строк и 4 колонки, мало, тк в реальных системах во много раз больше)_

**Ячейка А5.** Тип каждой колонки. `object` или `str` — так pandas называет текст.

In [36]:
df.dtypes

timestamp    str
user         str
ip           str
result       str
dtype: object

**Вопрос А5.** Колонка `timestamp` — число или текст? Почему это важно для модели?

**Ответ:** _(timestamp это текстовая колонка.Важно, потому что модель не может напрямую использовать дату и время как обычное числовое значение. Из времени сначала нужно получить полезные признаки, например час суток или день недели.)_

**Ячейка А6.** Сколько раз встречается каждое значение в колонке `result`.

In [37]:
df["result"].value_counts()

result
failure    36
success    33
Name: count, dtype: int64

**Вопрос А6.** Что бросается в глаза в соотношении успехов и неудач?

**Ответ:** _(Неудачных попуток больше чем успешных)_

**Ячейка А7.** Доля неудачных попыток.

In [38]:
len(df[df["result"] == "failure"]) / len(df)

0.5217391304347826

**Вопрос А7.** Прочитайте выражение изнутри наружу: что делает `df["result"] == "failure"`, что делает `df[...]`, что делает `len(...)`?

**Ответ:** _(52,2 %, df["result"] == "failure -проверяет является ли результат failure, df[...] -держит строки, где условие = истинна, len(...) =считает количество таких строк)_

**Ячейка А8.** С каких адресов больше всего неудач (топ-10).

In [39]:
df[df["result"] == "failure"]["ip"].value_counts().head(10)

ip
45.155.205.12     9
185.220.101.34    8
91.240.118.7      7
8.8.8.8           2
10.0.1.11         1
10.0.1.12         1
10.0.1.13         1
10.0.1.14         1
203.0.113.15      1
198.51.100.20     1
Name: count, dtype: int64

**Вопрос А8.** Чем первые три адреса отличаются от остальных? Что происходит? Как назвать это в отчёте об инциденте?

**Ответ:** _(пришло больше неудачных попыток, перебор паролей, это можно назвать аткой с методом перебора паролей (brute force))_

**Ячейка А9.** Новый признак: час суток. Затем — в какие часы происходят неудачи.

In [40]:
df["hour"] = pd.to_datetime(df["timestamp"]).dt.hour
df[df["result"] == "failure"]["hour"].value_counts().sort_index()

hour
1     19
2     10
8      2
9      2
10     2
11     1
Name: count, dtype: int64

**Вопрос А9.** В какие часы неудачи? Почему это подозрительно? Назовите два признака для будущей модели, которых **не было** в файле.

**Ответ:** _(1:00 - 19, 2:00 - 10, 8:00 - 2, 9:00 - 2. В это время меньше активность. Ночь или день)_

## А3. Глубже

**Ячейка А10.** Что происходило с учётной записью `admin`.

In [41]:
df[df["user"] == "admin"]["result"].value_counts()

result
failure    9
success    4
Name: count, dtype: int64

**Вопрос А10.** Сколько успешных входов под `admin`? Откуда и когда они были? Это попытка или уже инцидент?

Подсказка: `df[(df["user"] == "admin") & (df["result"] == "success")]` — знак `&` это «и», каждое условие в своих скобках.

**Ответ:** _(успешных 4, неуспешных 9. Это инцидент, тк попытка входа приходила со вгешнего айпишника)_

In [42]:
df[(df["user"] == "admin") & (df["result"] == "success")]


,timestamp,user,ip,result,hour
45,2025-03-11 02:17:00,admin,45.155.205.12,success,2
46,2025-03-11 02:19:00,admin,185.220.101.34,success,2
59,2025-03-11 16:06:00,admin,10.0.1.10,success,16
60,2025-03-11 16:55:00,admin,10.0.1.10,success,16


**Ячейка А11.** Только внутренние адреса (сеть `10.0.1.x`).

In [43]:
df[df["ip"].str.startswith("10.0.1.")]["result"].value_counts()

result
success    31
failure     7
Name: count, dtype: int64

**Вопрос А11.** Какая доля неудач внутри сети? Какой вывод про сотрудников?

**Ответ:** _(7 неудачных ил 36, это 19 процентов.Сотрудники лодыри, и много неудачных логинов пришло с внешних айпи адресов)_

**Ячейка А12.** Под какие имена пользователей подбирали пароль.

In [44]:
df[df["result"] == "failure"]["user"].value_counts().head(10)

user
admin            9
root             5
administrator    5
test             5
user             3
ivanov           2
petrov           2
sidorov          2
guest            2
smirnov          1
Name: count, dtype: int64

**Вопрос А12.** Почему атакующий перебирал именно эти имена?

**Ответ:** _(Они есть в каждой системе и это наиболее узнаваемые логины)_

---
# Часть Б. Пять задач → тип задачи машинного обучения

Четыре типа: **классификация** (на выходе категория), **регрессия** (число),
**кластеризация** (группы без названий), **поиск аномалий** («не похоже на обычное»).
Первые два — с учителем (в выборке есть правильный ответ), вторые два — без.

Заполните таблицу. Формула ответа: **тип · что на выходе · какие данные нужны · есть ли метка**.
Таблицу можно редактировать прямо здесь (два щелчка по ячейке).

| № | Задача | Тип | Что на выходе | Какие данные | Метка есть? |
|---|---|---|---|---|---|
| 1 | По тексту письма определить, фишинговое ли оно |классификация|фишинг или обычное письмо|текст|да|
| 2 | Разбить 10 000 алертов SIEM за месяц на группы похожих, чтобы аналитик разбирал группы |Кластеризация|группы алертов|Данныу о алертах SIEM|нет|
| 3 | Предсказать, через сколько дней заполнится диск сервера журналов |регрессия|Кол-во дней до заполнения диска|история заполнения диска|да|
| 4 | Сотрудник вошёл в 3 ночи с нового устройства и скачал 5 ГБ. Система должна заметить «не похоже на него» |поиск аномалий|Обычное или необычное поведение|Время входа, обьем скачанных данных, история поведения|нет|
| 5 | По описанию уязвимости оценить её опасность |классификация|категория опасности уязвимости|Описание и характеристики уязвимостей|да|

**Вопрос Б6.** Задача «по данным DLP решить, увольнять ли сотрудника». Что с ней не так с точки зрения машинного обучения?

**Ответ:** _(По данным DLP нельзя автоматически решать, увольнять сотрудника или нет. Нужна правильная постановка задачи, понятные признаки и надёжная метка, а также учёт контекста действий сотрудника.)_

**Вопрос Б7 (по плану курса).** Одна задача защиты информации с вашего места практики
(или из любой знакомой организации). Опишите её той же формулой: тип, что на выходе,
какие данные нужны, есть ли метка и откуда её взять. Три–пять предложений.

**Ответ:** _(нужны даные dlp, например прописывают флешки, то есть сертифицируют на данном предприятии)_

---
# Часть В. Домашняя практика: меняем код под свой вопрос

Все задания решаются **изменением** ячеек из части А, писать с нуля ничего не надо.
В каждом задании подсказка, какую ячейку взять за основу. Ответ — код в ячейке
плюс одно предложение словами.

## В1. Кто из сотрудников чаще всех ошибается паролем

Только внутренние адреса (`10.0.1.x`), только неудачи, топ-5 пользователей.
За основу: ячейки А11 (фильтр по адресу) и А12 (счёт по пользователям).
Два условия соединяются через `&`, каждое в своих скобках.

In [45]:
# В1


**Вывод В1 словами:** _(кто и сколько раз; это нормально?)_

## В2. Когда работал каждый из трёх внешних адресов

Для каждого из адресов `45.155.205.12`, `185.220.101.34`, `91.240.118.7` — первая и
последняя попытка по времени. Новая конструкция: `.groupby("ip")["timestamp"].agg(["min", "max"])`
считает минимум и максимум внутри каждой группы. Сначала отфильтруйте только
внешние адреса: `~df["ip"].str.startswith("10.0.1.")` — знак `~` это «не».

In [46]:
# В2


**Вывод В2 словами:** _(сколько минут длилась каждая атака? в один ли день?)_

## В3. Самый нагруженный день

Сколько попыток входа было в каждый день, и в какой день больше всего.
Новый признак — дата: `df["date"] = pd.to_datetime(df["timestamp"]).dt.date`
(как час в ячейке А9, только `.dt.date`). Дальше — `value_counts()`.

In [47]:
# В3


**Вывод В3 словами:** _(какой день и почему он выделяется?)_

## В4. Таблица признаков для будущей модели

Добавьте в `df` колонку `is_external`: 1, если адрес **не** начинается с `10.0.1.`,
иначе 0. Подсказка: `(~df["ip"].str.startswith("10.0.1.")).astype(int)`.
Затем посчитайте, сколько строк, где `is_external == 1` **и** `hour < 6`,
и сколько из них неудачных. Покажите `df.head()` — в таблице должны появиться
колонки `hour` и `is_external`.

In [48]:
# В4


**Вывод В4 словами:** _(сколько ночных внешних попыток; почему именно эти две колонки — признаки для модели?)_

## В5. Сохранить результат в файл

Топ-10 адресов по неудачам (ячейка А8) сохраните в файл `top_failed_ips.csv`:
допишите к выражению `.to_csv("top_failed_ips.csv")`. Затем откройте файл
обратно: `pd.read_csv("top_failed_ips.csv")` — убедитесь, что там 10 строк.
Файл положите в репозиторий рядом с блокнотом.

In [49]:
# В5


## В6. Свой вопрос к данным

Придумайте свой вопрос к журналу, на который ещё не отвечали, и ответьте кодом.
Примеры: с какого внутреннего адреса больше всего успешных входов; сколько разных
пользователей заходило с каждого внешнего адреса; есть ли сотрудники, которые
входили ночью с внутреннего адреса.

**Мой вопрос:** _(напишите здесь)_

In [50]:
# В6


**Ответ словами:** _(напишите здесь)_

---
# Как сдавать

1. Сохраните блокнот (Ctrl + S). Убедитесь, что **все ячейки выполнены и вывод виден**:
   сверху кнопка **Run All**, потом Ctrl + S. Проверяться будет вывод, а не только код.
2. Переименуйте файл: `ДЗ-03-Фамилия.ipynb`.
3. Залейте на GitHub любым из двух способов.

**Способ 1, без установки git (через браузер).** github.com → Sign up, если нет
аккаунта → зелёная кнопка **New** (репозиторий) → имя `tech-ai-2026`, Public →
Create repository → **Add file → Upload files** → перетащите `ДЗ-03-Фамилия.ipynb`,
`auth_log.csv` и `top_failed_ips.csv` → **Commit changes**.

**Способ 2, из VS Code.** Слева значок Source Control (Ctrl + Shift + G) →
**Publish to GitHub** → Public → VS Code попросит войти в GitHub. Нужен
установленный git (git-scm.com).

4. Ссылку на репозиторий вида `https://github.com/ваш-логин/tech-ai-2026` —
   преподавателю в мессенджер группы.

Что оценивается: части А и Б — выполнены все ячейки и заполнены все ответы;
часть В — код работает, вывод соответствует вопросу, есть вывод словами.
В1–В5 обязательны, В6 — плюс балл.